## Initializing the functions:

In [0]:
%run ../../Configurations/Init_Scripts

## Defining the source and destination paths:

In [0]:
dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")
 
dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
 
dbutils.widgets.text('ExternalLocation_silver',"/mntprod_silver")
ExternalLocation_silver = dbutils.widgets.get('ExternalLocation_silver')

In [0]:
# Source path for AccountCustomer_Dim file:
#-----------account_customer_path = '/mnt/silver/DIMCustomerMaster/'
account_customer_path = f'{ExternalLocation_silver}/DIMCustomerMaster/'
## destination folder path:
#------------dst_folder = '/mnt/silver/DIMAlignmentData'
dst_folder = f'{ExternalLocation_silver}/DIMAlignmentData'
# print(dst_folder)

dbutils.widgets.text('ConfigId','12')
ConfigId = dbutils.widgets.get('ConfigId')

dbutils.widgets.text('SourceTypeId','1')
SourceTypeId = dbutils.widgets.get('SourceTypeId')

# CreatedBy = 'ADB_PDM_Alignment'
# DeviceTypeCd = 'PDM'
# MessageTypeCd = 'PDMAlignment'

CreatedBy = dbutils.widgets.text("CreatedBy","ADB_PDM_Alignment")
CreatedBy = dbutils.widgets.get('CreatedBy')
DeviceTypeCd = dbutils.widgets.text("DeviceTypeCd","PDM")
DeviceTypeCd = dbutils.widgets.get('DeviceTypeCd')
MessageTypeCd = dbutils.widgets.text("MessageTypeCd","PDMAlignment")
MessageTypeCd = dbutils.widgets.get('MessageTypeCd')


#ingestion_related
IngestionConfigId = dbutils.widgets.text("IngestionConfigId","1")
IngestionConfigId = dbutils.widgets.get('IngestionConfigId')
SourceContainerPath = dbutils.widgets.text("SourceContainerPath","agn-dw-prod-ingest")
SourceContainerPath = dbutils.widgets.get('SourceContainerPath')
SourceFolderPath = dbutils.widgets.text("SourceFolderPath","outbound/CoolConnect")
SourceFolderPath = dbutils.widgets.get('SourceFolderPath')
DestinationContainerPath = dbutils.widgets.text("DestinationContainerPath","raw")
DestinationContainerPath = dbutils.widgets.get('DestinationContainerPath')
DestinationFolderPath = dbutils.widgets.text("DestinationFolderPath","SnowFlake/AlignmentData/CustomerAlignment")
DestinationFolderPath = dbutils.widgets.get('DestinationFolderPath')

Job_id=dbutils.widgets.text("Job_id","-1")
Job_id=dbutils.widgets.get("Job_id")
run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

## Copy the file from S3 Bucket into Raw Container ( Snowflake Folder)

In [0]:
files = dbutils.fs.ls("/mnt/s3_pdm/")
files_df = spark.createDataFrame(files)
files_df = files_df.orderBy(col('modificationTime').desc())
filtered_df = files_df.filter(col("path").contains("CoolConnect_Alignment"))
sourcefile = filtered_df.select('path').first()[0]
sf = sourcefile.split('/')[3]
#---------------------destination_path = f'/mnt/raw/{DestinationFolderPath}'
destination_path = f'{ExternalLocation_raw}/{DestinationFolderPath}'

try:
    dbutils.fs.cp(sourcefile, destination_path)
    processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':SourceContainerPath,'SourceFolderPath':SourceFolderPath,'SourceFileName':sf,
                     'DestinationContainerPath':DestinationContainerPath,'DestinationFolderPath':DestinationFolderPath,'DestinationFileName':sf,'PipelineStatus':"Succeeded",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':"PDM"
                      }]) 
    logIntoIngestionLogTable(processedFileList,'ADB_PDM_Alignment') 
except Exception as e:
    processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':SourceContainerPath,'SourceFolderPath':SourceFolderPath,'SourceFileName':sf,
                     'DestinationContainerPath':DestinationContainerPath,'DestinationFolderPath':DestinationFolderPath,'PipelineStatus':"Failed",'PipelineRunId':run_id, 'JobId':job_id, 'ErrorMessage':"Missing Source File",'DeviceType':"PDM"
                      }]) 
    logIntoIngestionLogTable(processedFileList,'ADB_PDM_Alignment') 
    # raise

## Processing the file:

In [0]:
## Source file paths:
#--------------src_folder = getLastFile('/mnt/raw/SnowFlake/AlignmentData/CustomerAlignment/') #ZLTQ_EQUIPMENT_MASTER_*
src_folder = getLastFile(f'{ExternalLocation_raw}/SnowFlake/AlignmentData/CustomerAlignment/') #ZLTQ_EQUIPMENT_MASTER_*

In [0]:
df_alignment_raw = (spark.read
                    .format('csv')
                    .options(header='True', delimiter='|')
                    .load(src_folder)
                    .select('*',col('_metadata.file_path').alias('SourceFilePath')
                               ,col("_metadata.file_name").alias('SourceFileName')
                               ,col("_metadata.file_modification_time").alias('file_modification_time')
                               ,col("_metadata.file_size").alias('SourceFileSize'))
                    .withColumn('SourceFilePath', regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''))
                    .withColumn('ConfigId', lit(ConfigId).cast('int'))
                       .withColumn('SourceTypeId', lit(SourceTypeId).cast('int'))
                       .withColumn("DeviceType",lit(DeviceTypeCd))
                       .withColumn("LogType",lit(MessageTypeCd))
                       .withColumn("DeviceId",lit(None).cast('string'))
                    )


df_customer_dim = (spark.read.format('delta').option('header', True)
                   .load(account_customer_path)
                   .select('AccountID','AccountPrimaryNm','AccountAddressId','AccountCityNm','AcctStateProvinceCd','AccountCountryAlpha2Cd')
                  )

In [0]:
# Read logsourcefileprocessed from silver zone:
logsourcefileprocessed_df = (spark.read.format('delta')
                             .option('header', True)
                             .load(src_filesProcessed)
                             .filter("LogFileStatus = 'InProgress'")
                             .select('SourceFilePath','FileNameUUID')) 

In [0]:
try:
    # loadAuditTables_DIM_InProgress(df_alignment_renamed,dst_folder,12,1,'ADB_PDM_Alignment')
    loadAuditTables_Ingestion_Log(df_alignment_raw,dst_folder,CreatedBy,'InProgress','')
       
    microBatchOutputDFF = (df_alignment_raw.groupBy("SourceFilePath","SourceFileName","SourceFileSize","ConfigId","SourceTypeId","DeviceId","file_modification_time").count()
                          .withColumnRenamed("count",'RecdCnt')
                          .withColumn("FileNameDeviceTypeCd",lit(DeviceTypeCd))
                          .withColumn("ExternalSerialNbr",lit(None))
                          .withColumn("InternalSerialNbr",lit(None))
                          .withColumn("FileNameMessageTypeCd",lit(MessageTypeCd))
                          .withColumn("FileNameDtTmstmp",date_format('file_modification_time', 'yyyyMMddHHmmss'))
                          .withColumn("FileNameApplicatorPortCd",lit(None))
                          .withColumn("FileNameCycleNbr",lit(None))
                          .withColumn('LogStartDate',lit(None))
                          .withColumn('LogEndDate',lit(None)).drop('file_modification_time'))
        
    loadlogProcessesDeltaTable(microBatchOutputDFF,dst_folder,CreatedBy,'InProgress','')
    # Renaming the columns:
    df_alignment_renamed = (df_alignment_raw
                        .withColumnRenamed('CUSTOMER_ID', 'AccountID')
                        .withColumnRenamed('TERRITORY_CODE', 'SalesTerritoryCd')
                        .withColumnRenamed('TERRITORY_DESCRIPTION', 'SalesTerritoryDesc')
                        .withColumnRenamed('TERRITORY_TYPE', 'SalesTerritoryTypeCd')
                        .withColumnRenamed('REP_WORKDAY_ID', 'SalesRepWorkdayID')
                        .withColumnRenamed('REP_FIRST_NAME', 'SalesRepFirstNm')
                        .withColumnRenamed('REP_LAST_NAME', 'SalesRepLastNm')
                        .withColumnRenamed('REP_EMAIL', 'SalesRepEmail')
                        .withColumnRenamed('REP_LOGIN_ID', 'SalesRepWorkdayLoginID')
                        .withColumnRenamed('AREA_CODE', 'SalesAreaCd')
                        .withColumnRenamed('AREA_DESCRIPTION', 'SalesAreaDesc')
                        .withColumnRenamed('AREA_MANAGER_WORKDAY_ID', 'SalesAreaMngrWorkdayID')
                        .withColumnRenamed('AREA_MANAGER_FIRST_NAME', 'SalesAreaMngrFirstNm')
                        .withColumnRenamed('AREA_MANAGER_LAST_NAME', 'SalesAreaMngrLastNm')
                        .withColumnRenamed('AREA_MANAGER_LOGIN_ID', 'SalesAreaMngrWorkdayLoginID')
                        .withColumnRenamed('AREA_MANAGER_EMAIL', 'SalesAreaMngrEmail')
                        .withColumnRenamed('REGION_CODE', 'SalesRegionCd')
                        .withColumnRenamed('REGION_DESCRIPTION', 'SalesRegionDesc')
                        .withColumnRenamed('REGION_MANAGER_WORKDAY_ID', 'SalesRegionMngrWorkdayID')
                        .withColumnRenamed('REGION_MANAGER_FIRST_NAME', 'SalesRegionMngrFirstNm')
                        .withColumnRenamed('REGION_MANAGER_LAST_NAME', 'SalesRegionMngrLastNm')
                        .withColumnRenamed('REGION_MANAGER_LOGIN_ID', 'SalesRegionMngrLoginID')
                        .withColumnRenamed('REGION_MANAGER_EMAIL', 'SalesRegionMngrEmail')
                        .withColumnRenamed('SALES_REP_PREFERRED_NAME', 'SalesRepPreferredNm')
                        .withColumnRenamed('AREA_MANAGER_PREFERRED_NAME', 'SalesAreaMngrPreferredNm')
                        .withColumnRenamed('REGION_MANAGER_PREFERRED_NAME', 'SalesRegionMngrPreferredNm')
                        .withColumn("CreatedBy",lit("ADB_PDM_Alignment"))
                        .withColumn("CreatedDt",current_timestamp())
                        .withColumn("UpdatedBy",lit("ADB_PDM_Alignment"))
                        .withColumn("UpdatedDt",current_timestamp())
                       )
    
    # Filterig the territory code:

    df_SalesTerritory = (df_alignment_renamed.filter(col('SalesTerritoryCd').like('CP%'))
                         .groupBy('AccountID').agg(min('SalesTerritoryCd').alias('SalesTerritoryCd')))

    # Self Join and grouping the data:
    join_condition = ['AccountID', 'SalesTerritoryCd']

    df_alignment = df_alignment_renamed.join(df_SalesTerritory, join_condition,  'inner').dropDuplicates()

    # Performing join with AccountCustomer_DIM:
    df_snowFlakeCustalgn_dim = df_alignment.join(df_customer_dim, 'AccountID', how = 'left')

    df_snowFlakeCustalgn_dim = (df_snowFlakeCustalgn_dim
                                .withColumn('AccountAddressId',col('AccountAddressId').cast('string'))
                                .withColumn('SalesRepWorkdayID',col('SalesRepWorkdayID').cast('string'))
                                .withColumn('SalesAreaMngrWorkdayID',col('SalesAreaMngrWorkdayID').cast('string'))
                                .withColumn('SalesRegionMngrWorkdayID',col('SalesRegionMngrWorkdayID').cast('string'))
                                )

    df_snowFlakeCustalgn_dim_UUID = df_snowFlakeCustalgn_dim.join(logsourcefileprocessed_df,"SourceFilePath",'inner')
    
    # df_snowFlakeCustalgn_dim_UUID.display()
    # Writing in delta format:
    # # Inserting new records
    # srcDupCount = df_snowFlakeCustalgn_dim.groupBy('AccountID').count().filter('count>1').count()
    # assert srcDupCount == 0, "Duplicate records found in Snow Flake Allignment Data"

    
    Deltatbl_PDmAlignment = DeltaTable.forPath(spark, dst_folder)  
    (Deltatbl_PDmAlignment.alias("tgt")
            .merge(
            df_snowFlakeCustalgn_dim_UUID.alias("src"),
                "tgt.AccountID = src.AccountID")
      .whenNotMatchedInsert(values =
      {
        "tgt.FileNameUUID" : "src.FileNameUUID",
        "tgt.AccountID" : "src.AccountID",
        "tgt.AccountPrimaryNm" : "src.AccountPrimaryNm",
        "tgt.AccountAddressId" : "src.AccountAddressId",
        "tgt.AccountCityNm" : "src.AccountCityNm",
        "tgt.AcctStateProvinceCd" : "src.AcctStateProvinceCd",
        "tgt.AccountCountryAlpha2Cd" : "src.AccountCountryAlpha2Cd",
        "tgt.SalesTerritoryCd" : "src.SalesTerritoryCd",
        "tgt.SalesTerritoryDesc" : "src.SalesTerritoryDesc",
        "tgt.SalesTerritoryTypeCd" : "src.SalesTerritoryTypeCd",
        "tgt.SalesRepWorkdayID" : "src.SalesRepWorkdayID",
        "tgt.SalesRepFirstNm" : "src.SalesRepFirstNm",
        "tgt.SalesRepLastNm" : "src.SalesRepLastNm",
        "tgt.SalesRepEmail" : "src.SalesRepEmail",
        "tgt.SalesRepWorkdayLoginID" : "src.SalesRepWorkdayLoginID",
        "tgt.SalesAreaCd" : "src.SalesAreaCd",
        "tgt.SalesAreaDesc" : "src.SalesAreaDesc",
        "tgt.SalesAreaMngrWorkdayID" : "src.SalesAreaMngrWorkdayID",
        "tgt.SalesAreaMngrFirstNm" : "src.SalesAreaMngrFirstNm",
        "tgt.SalesAreaMngrLastNm" : "src.SalesAreaMngrLastNm",
        "tgt.SalesAreaMngrWorkdayLoginID" : "src.SalesAreaMngrWorkdayLoginID",
        "tgt.SalesAreaMngrEmail" : "src.SalesAreaMngrEmail",
        "tgt.SalesRegionCd" : "src.SalesRegionCd",
        "tgt.SalesRegionDesc" : "src.SalesRegionDesc",
        "tgt.SalesRegionMngrWorkdayID" : "src.SalesRegionMngrWorkdayID",
        "tgt.SalesRegionMngrFirstNm" : "src.SalesRegionMngrFirstNm",
        "tgt.SalesRegionMngrLastNm" : "src.SalesRegionMngrLastNm",
        "tgt.SalesRegionMngrLoginID" : "src.SalesRegionMngrLoginID",
        "tgt.SalesRegionMngrEmail" : "src.SalesRegionMngrEmail",
        "tgt.SalesRepPreferredNm" : "src.SalesRepPreferredNm",
        "tgt.SalesAreaMngrPreferredNm" : "src.SalesAreaMngrPreferredNm",
        "tgt.SalesRegionMngrPreferredNm" : "src.SalesRegionMngrPreferredNm",
        "tgt.CreatedBy" : "src.CreatedBy",
        "tgt.CreatedDt" : "src.CreatedDt",
        "tgt.UpdatedBy" : "src.UpdatedBy",
        "tgt.UpdatedDt" : "src.UpdatedDt"
      })
      .whenMatchedUpdate(
          #condition = "tgt.SalesTerritoryCd <> src.SalesTerritoryCd",
          set =
      {
        "tgt.FileNameUUID" : "src.FileNameUUID",
        "tgt.AccountID" : "src.AccountID",
        "tgt.AccountPrimaryNm" : "src.AccountPrimaryNm",
        "tgt.AccountAddressId" : "src.AccountAddressId",
        "tgt.AccountCityNm" : "src.AccountCityNm",
        "tgt.AcctStateProvinceCd" : "src.AcctStateProvinceCd",
        "tgt.AccountCountryAlpha2Cd" : "src.AccountCountryAlpha2Cd",
        "tgt.SalesTerritoryCd" : "src.SalesTerritoryCd",
        "tgt.SalesTerritoryDesc" : "src.SalesTerritoryDesc",
        "tgt.SalesTerritoryTypeCd" : "src.SalesTerritoryTypeCd",
        "tgt.SalesRepWorkdayID" : "src.SalesRepWorkdayID",
        "tgt.SalesRepFirstNm" : "src.SalesRepFirstNm",
        "tgt.SalesRepLastNm" : "src.SalesRepLastNm",
        "tgt.SalesRepEmail" : "src.SalesRepEmail",
        "tgt.SalesRepWorkdayLoginID" : "src.SalesRepWorkdayLoginID",
        "tgt.SalesAreaCd" : "src.SalesAreaCd",
        "tgt.SalesAreaDesc" : "src.SalesAreaDesc",
        "tgt.SalesAreaMngrWorkdayID" : "src.SalesAreaMngrWorkdayID",
        "tgt.SalesAreaMngrFirstNm" : "src.SalesAreaMngrFirstNm",
        "tgt.SalesAreaMngrLastNm" : "src.SalesAreaMngrLastNm",
        "tgt.SalesAreaMngrWorkdayLoginID" : "src.SalesAreaMngrWorkdayLoginID",
        "tgt.SalesAreaMngrEmail" : "src.SalesAreaMngrEmail",
        "tgt.SalesRegionCd" : "src.SalesRegionCd",
        "tgt.SalesRegionDesc" : "src.SalesRegionDesc",
        "tgt.SalesRegionMngrWorkdayID" : "src.SalesRegionMngrWorkdayID",
        "tgt.SalesRegionMngrFirstNm" : "src.SalesRegionMngrFirstNm",
        "tgt.SalesRegionMngrLastNm" : "src.SalesRegionMngrLastNm",
        "tgt.SalesRegionMngrLoginID" : "src.SalesRegionMngrLoginID",
        "tgt.SalesRegionMngrEmail" : "src.SalesRegionMngrEmail",
        "tgt.SalesRepPreferredNm" : "src.SalesRepPreferredNm",
        "tgt.SalesAreaMngrPreferredNm" : "src.SalesAreaMngrPreferredNm",
        "tgt.SalesRegionMngrPreferredNm" : "src.SalesRegionMngrPreferredNm",
        #"tgt.CreatedBy" : "src.CreatedBy",
        #"tgt.CreatedDt" : "src.CreatedDt",
        "tgt.UpdatedBy" : "src.UpdatedBy",
        "tgt.UpdatedDt" : "src.UpdatedDt"
      })
      .execute()
    )
    
    df_alignment_raw1 = (df_snowFlakeCustalgn_dim_UUID.withColumn('ConfigId',lit(ConfigId)).groupBy('SourceFilePath','SourceFileName','FileNameUUID','ConfigId')
                                    .agg(min(col('file_modification_time')).alias('LogStartDate'),
                                         max(col('file_modification_time')).alias('LogEndDate')))
    # loadAuditTables_DIM_Complete(df_alignment_renamed,dst_folder,12,1,'ADB_PDM_Alignment','Succeeded','')
    loadlogProcessesDeltaTable(df_alignment_raw1,dst_folder,CreatedBy,'Succeeded','')
    loadAuditTables_Ingestion_Log(df_alignment_raw,dst_folder,CreatedBy,'Succeeded','')
except Exception as e:
    print(str(e))
    # loadAuditTables_DIM_Complete(df_alignment_renamed,dst_folder,12,1,'ADB_PDM_Alignment','Failed',str(e))
    loadlogProcessesDeltaTable(df_alignment_raw,dst_folder,CreatedBy,'Failed',str(e))
    loadAuditTables_Ingestion_Log(df_alignment_raw,dst_folder,CreatedBy,'Failed',str(e))
    raise